# Chicago Crimes - Pseudo-Labeling (Risk Score 0-100)
Membangun label risiko per sel grid x bulan menggunakan:
- Severity weighting = tingkat bahaya (Primary Type) x modifier keparahan (Description)
- Temporal decay = kejadian lama makin kecil bobotnya (half-life 3 bulan)
- Spatial decay = risiko menyebar ke sel tetangga (kernel Gaussian)
Output: pseudo_labels_gridmonth.parquet & training_table.parquet. 

## 0. Setup

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
warnings.filterwarnings('ignore')
pd.set_option('display.width',180); pd.set_option('display.max_columns',60)
BRAND_INDIGO='#3D2B9E'; BRAND_VIOLET='#7B3FE4'; BRAND_MAGENTA='#CE1C8E'; BRAND_PINK='#EC6DB4'; BRAND_LIGHT='#F7B4DA'
BRAND_COLORS=[BRAND_MAGENTA,BRAND_INDIGO,BRAND_VIOLET,BRAND_PINK,BRAND_LIGHT]; ACCENT=BRAND_MAGENTA
BRAND_CMAP=LinearSegmentedColormap.from_list('brand',['#241663',BRAND_INDIGO,BRAND_VIOLET,BRAND_MAGENTA,BRAND_PINK,BRAND_LIGHT])
try: mpl.colormaps.register(BRAND_CMAP)
except (ValueError,AttributeError): pass
plt.rcParams['axes.prop_cycle']=plt.cycler(color=BRAND_COLORS); plt.rcParams['image.cmap']='brand'
plt.rcParams['figure.figsize']=(12,5); plt.rcParams['figure.dpi']=110; plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=0.25
print('pandas',pd.__version__,'| numpy',np.__version__,'| mode CPU')

pandas 2.3.3 | numpy 2.0.2 | mode CPU


## 1. Konfigurasi

In [2]:
FE_DIR='/kaggle/input/notebooks/fransiskaadin/04-feature-engineering-chicago-crimes'
OUT_DIR='/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
GRID=0.01
TEMPORAL_HALFLIFE=3.0
TEMPORAL_WINDOW=12
SPATIAL_BANDWIDTH=1.0
SPATIAL_RADIUS=2
CLIP_PCTL=99.5
TIER_BINS=[-0.01,50,80,95,100.01]
TIER_LABELS=['LOW','MEDIUM','HIGH','VERY HIGH']
SEVERITY={}
for t,names in [
  (16,['HOMICIDE','CRIMINAL SEXUAL ASSAULT','CRIM SEXUAL ASSAULT','KIDNAPPING','HUMAN TRAFFICKING']),
  (8,['ROBBERY','ARSON','WEAPONS VIOLATION','OFFENSE INVOLVING CHILDREN','CONCEALED CARRY LICENSE VIOLATION','SEX OFFENSE','CRIMINAL SEXUAL ABUSE']),
  (4,['BATTERY','ASSAULT','BURGLARY','NARCOTICS','INTIMIDATION','STALKING','PROSTITUTION','OTHER NARCOTIC VIOLATION']),
  (2,['THEFT','CRIMINAL DAMAGE','CRIMINAL TRESPASS','DECEPTIVE PRACTICE','MOTOR VEHICLE THEFT','PUBLIC PEACE VIOLATION','INTERFERENCE WITH PUBLIC OFFICER','LIQUOR LAW VIOLATION'])]:
    for n in names: SEVERITY[n]=t
print('FE_DIR',FE_DIR)
print('OUT_DIR',OUT_DIR,'| tier map size',len(SEVERITY))

FE_DIR /kaggle/input/notebooks/fransiskaadin/04-feature-engineering-chicago-crimes
OUT_DIR /kaggle/working | tier map size 28


## 2. Load features_incident

In [3]:
def find_load(name):
    cand=[FE_DIR+'/'+name+'.parquet','/kaggle/working/'+name+'.parquet',name+'.parquet']
    cand+=sorted(glob.glob('/kaggle/input/**/'+name+'.parquet', recursive=True))
    for p in cand:
        if os.path.exists(p):
            try: return pd.read_parquet(p),p
            except Exception: pass
    for p in [x.replace('.parquet','.csv') for x in cand]:
        if os.path.exists(p):
            return pd.read_csv(p, low_memory=False),p
    raise FileNotFoundError(name+' tidak ditemukan (parquet/csv)')
inc,src=find_load('features_incident')
print('Loaded features_incident:',src,'| shape',inc.shape)
print('Kolom:',list(inc.columns))

Loaded features_incident: /kaggle/input/notebooks/fransiskaadin/04-feature-engineering-chicago-crimes/features_incident.parquet | shape (8534663, 23)
Kolom: ['Date', 'period_str', 'year', 'month', 'hour', 'dow', 'primary_type_canon', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Latitude', 'Longitude', 'coord_missing', 'gi', 'gj', 'glat', 'glon', 'is_night', 'is_weekend', 'is_violent', 'is_property']


## 3. Severity: Primary Type + Description (UPGRADE)
Severity insiden = tier(Primary Type) x modifier(Description). Modifier menaikkan bobot untuk
senjata api / agravasi / senjata tajam, dan sedikit menurunkan untuk kasus ringan (SIMPLE/PETTY).
Contoh: ROBBERY + ARMED:HANDGUN > ROBBERY biasa; BATTERY + AGGRAVATED > BATTERY SIMPLE.

In [4]:
if 'primary_type_canon' not in inc.columns:
    inc['primary_type_canon']=inc['Primary Type'].astype('str').str.upper().str.strip()
if 'Description' not in inc.columns: inc['Description']=''
inc['sev_base']=inc['primary_type_canon'].map(SEVERITY).fillna(1.0).astype(float)
D=inc['Description'].fillna('').astype('str').str.upper()
gun=D.str.contains(r'HANDGUN|FIREARM|SHOTGUN|RIFLE|GUN|ARMED', regex=True)
agg=D.str.contains('AGGRAVATED', regex=True)
weap=D.str.contains(r'KNIFE|CUTTING|OTHER DANGEROUS WEAPON|WEAPON', regex=True)
simple=D.str.contains(r'SIMPLE|PETTY|[$]500 AND UNDER', regex=True)
mult=(np.where(gun,1.8,1.0)*np.where(agg,1.4,1.0)*np.where(weap,1.3,1.0)*np.where(simple,0.9,1.0))
mult=np.clip(mult,0.8,2.2)
inc['desc_mult']=mult
inc['severity']=inc['sev_base']*inc['desc_mult']
print('Dampak modifier Description pada severity (12 jenis teratas):')
chk=inc.groupby('primary_type_canon').agg(base=('sev_base','first'),final_mean=('severity','mean'),final_max=('severity','max'),n=('severity','size')).sort_values('final_mean',ascending=False).head(12)
print(chk.to_string())
print('Insiden dengan modifier != 1.0:', int((inc['desc_mult']!=1.0).sum()), '(', round(100*(inc['desc_mult']!=1.0).mean(),2),'% )')

Dampak modifier Description pada severity (12 jenis teratas):
                                   base  final_mean  final_max        n
primary_type_canon                                                     
CRIMINAL SEXUAL ASSAULT            16.0   22.703496       35.2    39650
KIDNAPPING                         16.0   16.400583       22.4     7541
HUMAN TRAFFICKING                  16.0   16.000000       16.0      147
HOMICIDE                           16.0   16.000000       16.0    14183
WEAPONS VIOLATION                   8.0   13.852711       14.4   126994
ROBBERY                             8.0   11.898396       17.6   316790
CONCEALED CARRY LICENSE VIOLATION   8.0    9.507393       14.4     1745
ARSON                               8.0    8.471350       11.2    14576
OFFENSE INVOLVING CHILDREN          8.0    8.203415       11.2    61321
SEX OFFENSE                         8.0    8.117296       11.2    34893
ASSAULT                             4.0    4.759157        8.8   573910
BA

Basis tier per Primary Type dikalikan modifier dari `Description`, dan hasilnya berperilaku persis seperti niat desain: `CRIMINAL SEXUAL ASSAULT` naik dari 16 ke rata-rata 22,7 (maks 35,2 saat AGGRAVATED), `ROBBERY` dari 8 ke maks 17,6 saat ARMED:HANDGUN, sedangkan `BATTERY` SIMPLE tetap rendah. Sebanyak 36,31% insiden terkena modifier != 1,0 membuktikan deskripsi benar-benar membedakan keparahan, bukan sekadar tipe. Inilah jawaban atas contoh scoring table di task (ROBBERY+ARMED > ROBBERY biasa).


## 4. Harm teramati per sel x bulan

In [5]:
for c in ['gi','gj']:
    inc[c]=pd.to_numeric(inc[c], errors='coerce')
if 'coord_missing' in inc.columns: inc=inc[~inc['coord_missing'].astype(bool)]
inc=inc[inc['gi'].notna() & inc['gj'].notna()]
if 'period_str' not in inc.columns:
    inc['period_str']=pd.to_datetime(inc['Date']).dt.to_period('M').astype('str')
harm=inc.groupby(['gi','gj','period_str'], observed=True).agg(
    harm=('severity','sum'), n_inc=('severity','size'), sev_mean=('severity','mean')).reset_index()
print('Harm teramati:', harm.shape, '| total harm', round(float(harm["harm"].sum()),1))

Harm teramati: (197608, 6) | total harm 29929467.2


Severity insiden dijumlahkan ke unit sel x bulan menjadi `harm` (total 29,9 jt pada 197.608 sel-bulan). Menjumlahkan severity - bukan sekadar menghitung jumlah kejahatan - membuat satu HOMICIDE berbobot jauh lebih besar daripada banyak THEFT ringan, sesuai tujuan mengukur risiko, bukan volume.


## 5. Load panel features_gridmonth & gabung harm

In [6]:
panel,src2=find_load('features_gridmonth')
print('Loaded features_gridmonth:',src2,'| shape',panel.shape)
for c in ['gi','gj']: panel[c]=pd.to_numeric(panel[c], errors='coerce')
panel=panel.merge(harm[['gi','gj','period_str','harm']], on=['gi','gj','period_str'], how='left')
panel['harm']=panel['harm'].fillna(0.0)
panel['period_dt']=pd.to_datetime(panel['period_str']+'-01')
panel=panel.sort_values(['gi','gj','period_dt']).reset_index(drop=True)
print('Panel + harm:', panel.shape, '| sel-bulan tanpa harm:', int((panel["harm"]==0).sum()))

Loaded features_gridmonth: /kaggle/input/notebooks/fransiskaadin/04-feature-engineering-chicago-crimes/features_gridmonth.parquet | shape (226784, 34)
Panel + harm: (226784, 36) | sel-bulan tanpa harm: 29176


`harm` ditempel ke panel 226.784 sel-bulan; 29.176 sel-bulan tanpa harm tetap dipertahankan bernilai 0. Ini penting: bulan-bulan sepi harus ikut dinilai agar Risk Score merepresentasikan ketiadaan risiko, bukan sekadar data yang hilang.


## 6. Temporal decay (half-life 3 bulan, trailing 12 bulan)
harm_temporal[t] = sum_{k=0..W} harm[t-k] * 0.5^(k/half_life).

In [7]:
W=TEMPORAL_WINDOW; hl=TEMPORAL_HALFLIFE
weights=[0.5**(k/hl) for k in range(W+1)]
gh=panel.groupby(['gi','gj'], observed=True)['harm']
ht=np.zeros(len(panel))
for k,w in enumerate(weights):
    ht=ht+gh.shift(k).fillna(0.0).to_numpy()*w
panel['harm_temporal']=ht
print('Bobot temporal k=0..12:', [round(w,3) for w in weights])
print('harm_temporal mean', round(float(panel["harm_temporal"].mean()),3), '| max', round(float(panel["harm_temporal"].max()),1))

Bobot temporal k=0..12: [1.0, 0.794, 0.63, 0.5, 0.397, 0.315, 0.25, 0.198, 0.157, 0.125, 0.099, 0.079, 0.062]
harm_temporal mean 604.589 | max 6272.7


**temporal decay (recency).** Bobot meluruh mengikuti half-life 3 bulan (1,0 -> 0,5 pada bulan ke-3 -> 0,062 pada bulan ke-12). Kejahatan bulan lalu dihargai jauh lebih tinggi daripada 12 bulan lalu, menjawab pertanyaan task "apakah kejadian 3 tahun lalu serelevan kemarin?". Half-life 3 bulan dipilih moderat karena EDA menunjukkan hotspot cukup stabil - tidak perlu peluruhan agresif.


## 7. Spatial decay (kernel Gaussian, bandwidth 1 sel, radius 2)
harm_spatial[cell] = sum_{tetangga} harm_temporal[tetangga] * exp(-(di^2+dj^2)/(2*bw^2)).

In [8]:
bw=SPATIAL_BANDWIDTH; R=SPATIAL_RADIUS
hs=panel.set_index(['gi','gj','period_str'])['harm_temporal']
gi=panel['gi'].to_numpy(); gj=panel['gj'].to_numpy(); per=panel['period_str'].to_numpy()
acc=np.zeros(len(panel)); wsum=0.0
for di in range(-R,R+1):
    for dj in range(-R,R+1):
        w=np.exp(-(di*di+dj*dj)/(2*bw*bw)); wsum+=w
        idx=list(zip(gi+di, gj+dj, per))
        vals=hs.reindex(idx).to_numpy()
        acc=acc+np.where(np.isnan(vals),0.0,vals)*w
panel['harm_spatial']=acc
print('Total bobot kernel', round(wsum,3))
print('harm_spatial mean', round(float(panel["harm_spatial"].mean()),3), '| max', round(float(panel["harm_spatial"].max()),1))

Total bobot kernel 6.169
harm_spatial mean 3639.116 | max 20336.1


**spatial decay (proximity).** Kernel Gaussian (bandwidth 1 sel, radius 2) menyebarkan harm ke sel tetangga dengan total bobot 6,17. Risiko sebuah lokasi kini turut dipengaruhi lingkungan sekitarnya - menjawab pertanyaan task "apakah hanya titik itu, atau area sekitar juga relevan?". Peluruhan berdasarkan jarak menjaga agar pengaruh melemah semakin jauh sebuah sel.


## 8. Risk Score 0-100 (log1p -> winsorize -> min-max)

In [9]:
panel['risk_raw']=panel['harm_spatial'].astype(float)
r=np.log1p(panel['risk_raw'].clip(lower=0))
cap=np.percentile(r, CLIP_PCTL); r=np.clip(r,0,cap)
span=float(r.max()-r.min())
panel['risk_score']=((r-r.min())/span*100.0) if span>0 else 0.0
panel['risk_tier']=pd.cut(panel['risk_score'], bins=TIER_BINS, labels=TIER_LABELS)
maxp=panel['period_str'].max()
panel['is_partial_period']=(panel['period_str']==maxp).astype(int)
print('risk_score mean', round(float(panel["risk_score"].mean()),2), '| median', round(float(panel["risk_score"].median()),2), '| skew', round(float(panel["risk_score"].skew()),3))
print('Distribusi tier:'); print(panel['risk_tier'].value_counts().to_string())
print('Korelasi risk_score vs harm', round(float(panel[["risk_score","harm"]].corr().iloc[0,1]),3), '| vs risk_raw', round(float(panel[["risk_score","risk_raw"]].corr().iloc[0,1]),3))

risk_score mean 78.02 | median 82.14 | skew -1.39
Distribusi tier:
risk_tier
HIGH         110603
MEDIUM        87473
VERY HIGH     16946
LOW           11762
Korelasi risk_score vs harm 0.627 | vs risk_raw 0.781


**Risk Score 0-100.** Pipeline `log1p -> winsorize p99,5 -> min-max` menjinakkan ekor ekstrem (skewness densitas 2,02 dari diagnostik) sebelum normalisasi. Hasil: mean 78,0 / median 82,1 dengan sebaran tier LOW/MEDIUM/HIGH/VERY HIGH. Distribusi condong ke atas wajar untuk kota padat kejahatan; korelasi kuat vs `risk_raw` (0,78) menandakan transformasi mempertahankan urutan risiko, bukan mengacaknya.


## 9. Simpan pseudo_labels_gridmonth & training_table

In [10]:
pl_cols=['gi','gj','period_str','glat','glon','n_crimes','harm','harm_temporal','harm_spatial','risk_raw','risk_score','risk_tier','is_partial_period']
pl_cols=[c for c in pl_cols if c in panel.columns]
pseudo=panel[pl_cols].copy()
add=pseudo[['gi','gj','period_str','risk_score','risk_tier','harm','harm_temporal','harm_spatial','risk_raw','is_partial_period']]
train=panel.drop(columns=['period_dt'], errors='ignore').merge(add, on=['gi','gj','period_str'], how='left', suffixes=('','_y'))
train=train.drop(columns=[c for c in train.columns if c.endswith('_y')])
def save(df,name):
    try:
        df.to_parquet(os.path.join(OUT_DIR,name+'.parquet'), index=False); print('Tersimpan',name+'.parquet',df.shape)
    except Exception as e:
        df.to_csv(os.path.join(OUT_DIR,name+'.csv'), index=False); print('Fallback CSV',name+'.csv ('+type(e).__name__+')',df.shape)
save(pseudo,'pseudo_labels_gridmonth')
save(train,'training_table')
print('pseudo cols:',list(pseudo.columns))
print('NaN target di training_table:', int(train['risk_score'].isna().sum()))

Tersimpan pseudo_labels_gridmonth.parquet (226784, 13)
Tersimpan training_table.parquet (226784, 41)
pseudo cols: ['gi', 'gj', 'period_str', 'glat', 'glon', 'n_crimes', 'harm', 'harm_temporal', 'harm_spatial', 'risk_raw', 'risk_score', 'risk_tier', 'is_partial_period']
NaN target di training_table: 0


Dua artefak tersimpan: `pseudo_labels_gridmonth` (226.784 x 13, label inti) dan `training_table` (226.784 x 41, fitur + label siap model), keduanya dengan **0 NaN pada target**. Kolom `is_partial_period` menandai bulan berjalan yang belum lengkap agar bisa dikecualikan saat training. 


## 10. Ringkasan

In [11]:
print('Rentang periode:', panel['period_str'].min(),'->',panel['period_str'].max())
print('Sel unik:', panel[['gi','gj']].drop_duplicates().shape[0])
top=panel.groupby(['glat','glon'], observed=True)['risk_score'].mean().sort_values(ascending=False).head(5)
print('Hotspot rata-rata risk_score tertinggi:'); print(top.to_string())

Rentang periode: 2001-01 -> 2026-04
Sel unik: 746
Hotspot rata-rata risk_score tertinggi:
glat   glon  
41.87  -87.72    98.197481
41.88  -87.72    97.901138
41.87  -87.71    97.706474
41.89  -87.63    97.678721
41.88  -87.73    97.578367


Sel ber-risk_score tertinggi (mis. 41,87,-87,72 dan 41,89,-87,63) konsisten dengan hotspot yang teridentifikasi di EDA, artinya label yang dibentuk tidak asal tinggi, melainkan menaruh risiko pada lokasi yang memang secara historis paling rawan. 
